本章实现了自注意机制、掩码注意力和多头掩码注意力。

In [1]:
import torch
from torch import nn

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, input_dim, output_dim, max_seq_len, bias=False, dropout=0.1):
        super(CausalAttention, self).__init__()
        self.w_q = nn.Linear(input_dim, output_dim, bias=bias)
        self.w_k = nn.Linear(input_dim, output_dim, bias=bias)
        self.w_v = nn.Linear(input_dim, output_dim, bias=bias)
        self.w_o = nn.Linear(output_dim, output_dim, bias=bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        )
    
    def forward(self, x):
        _, seq_len, _ = x.shape
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)
        attention_scores = q @ k.transpose(1, 2) / (k.shape[-1] ** 0.5)
        attention_scores.masked_fill_(
            self.mask[:seq_len, :seq_len], -torch.inf
        )
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        attention_output = attention_weights @ v
        return self.w_o(attention_output)

In [ ]:
# 为1的地方会被填充-inf
a = torch.triu(torch.ones(3, 3), diagonal=1)
a

tensor([[0., 1., 1.],
        [0., 0., 1.],
        [0., 0., 0.]])

In [11]:
causal = CausalAttention(4, 4, 4)
x = torch.randn(2, 4, 4)
out = causal(x)
print(out.shape)  # 应该是 (2, 4, 4)

torch.Size([2, 4, 4])


In [14]:
class MultiHeadAttention(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, max_seq_len, bias=False, dropout=0.1):
        super().__init__()
        self.head_dim = out_dim // num_heads
        self.num_heads = num_heads
        self.w_q = nn.Linear(in_dim, out_dim, bias=bias)
        self.w_k = nn.Linear(in_dim, out_dim, bias=bias)
        self.w_v = nn.Linear(in_dim, out_dim, bias=bias)
        self.w_o = nn.Linear(out_dim, out_dim, bias=bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
                torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        )
    
    def forward(self, x):
        bs, seq_len, _ = x.shape
        q = self.w_q(x).view(bs, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.w_k(x).view(bs, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.w_v(x).view(bs, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        attention_scores = q @ k.transpose(-1, -2) / (self.head_dim ** 0.5)
        attention_scores.masked_fill_(
            self.mask[:seq_len, :seq_len], -torch.inf
        )
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        attention_output = attention_weights @ v
        attention_output = attention_output.transpose(1, 2).contiguous().view(bs, seq_len, -1)
        return self.w_o(attention_output)


In [15]:
ma = MultiHeadAttention(4, 8, 2, 4)
x = torch.randn(2, 4, 4)
out = ma(x)
print(out.shape)  # 应该是 (2, 4, 8)

torch.Size([2, 4, 8])
